In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Pertemuan7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 09:40:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/09 09:40:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


In [3]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/krock/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/krock/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/krock/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/krock/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [5]:
# A. EXTRACT (bobot 10%)
# Baca ketiga sumber data dari HDFS (bukan disk lokal) menjadi tiga Spark 
# DataFrame.

df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/krock/tugas7/raw/tugas7_transaksi.csv",
    header=True, inferSchema=True
)
df_cabang = spark.read.json("hdfs://localhost:9000/user/krock/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/krock/tugas7/raw/tugas7_menu.csv",
    header=True, inferSchema=True
)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Transaksi: 4000 | Cabang: 3 | Menu: 15


In [6]:
# B. TRANSFORM — Join & Kolom Turunan (bobot 20%)
# Gabungkan ketiga tabel, tambahkan kolom total_penjualan (qty x harga).

df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                            .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)


+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [9]:
# C. TRANSFORM — Window Function (bobot 20%)
# Tentukan top-2 menu terlaris (berdasarkan total_penjualan) di setiap cabang, gunakan 
# row_number().

ringkasan_produk_toko = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top2_menu = ringkasan_produk_toko.withColumn("peringkat", row_number().over(window_cabang)) \
                                    .filter(col("peringkat") <=2)
top2_menu.orderBy("nama_cabang", "peringkat").show(20)

[Stage 19:>                                                         (0 + 1) / 1]

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



In [11]:
# D. TRANSFORM — Spark SQL (bobot 20%)
# Menggunakan Spark SQL murni, tampilkan total penjualan per kepala_barista, diurutkan
# dari tertinggi.

df_gabungan.createOrReplaceTempView("gabungan")

kontribusi_kepala_barista = spark.sql('''
        SELECT kota, kepala_barista, SUM(total_penjualan) AS total_penjualan
        FROM gabungan
        GROUP BY kota, kepala_barista
        ORDER BY total_penjualan DESC
''')
kontribusi_kepala_barista.show()

+----------+--------------+---------------+
|      kota|kepala_barista|total_penjualan|
+----------+--------------+---------------+
|  Semarang|         Nadia|       90542000|
|  Magelang|         Nadia|       87138000|
|Yogyakarta|          Reza|       84645000|
+----------+--------------+---------------+



In [13]:
# E. LOAD (bobot 15%)
# Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, 
# path /user/[username]/tugas7/processed/laporan, dipartisi berdasarkan kategori_menu. 
# Verifikasi dengan membaca kembali dan menghitung jumlah baris.

!hdfs dfs -mkdir -p /user/krock/tugas7/processed
df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/krock/tugas7/processed/laporan_cabang"
)

print("Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori produk.")
!hdfs dfs -ls /user/krock/tugas7/processed/laporan_cabang


Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori produk.
Found 4 items
-rw-r--r--   3 krock supergroup          0 2026-10-09 21:29 /user/krock/tugas7/processed/laporan_cabang/_SUCCESS
drwxr-xr-x   - krock supergroup          0 2026-10-09 21:29 /user/krock/tugas7/processed/laporan_cabang/kategori_menu=Kopi
drwxr-xr-x   - krock supergroup          0 2026-10-09 21:29 /user/krock/tugas7/processed/laporan_cabang/kategori_menu=Makanan Ringan
drwxr-xr-x   - krock supergroup          0 2026-10-09 21:29 /user/krock/tugas7/processed/laporan_cabang/kategori_menu=Non-Kopi


In [17]:
# F. Ringkasan Eksekutif & Rekomendasi (bobot 15%)
# Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai 
# transaksi), lalu tulis pada markdown cell (minimal 120 kata): cabang mana yang 
# berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan
# kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

df_final = spark.read.parquet("hdfs://localhost:9000/user/krock/tugas7/processed/laporan_cabang")

ringkasan_eksekutif = df_final.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan_cabang"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_cabang").desc())

ringkasan_eksekutif.show()




+--------------------+----------+--------------+----------------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan_cabang|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+----------------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|              90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|              87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|              84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+----------------------+----------------+-------------------------+



In [ ]:
Dari data yang didapat, cabang yang memiliki kinerja terbaik adalah Kopi Nusantara cabang Semarang dengan total penjualan cabang
tertinggi sebesar 90.542.000 dengan jumlah transaksi sebanyak 1393 transaksi. Lalu untuk cabang dengan kinerja terlemah adalah cabang
yogyakarta dengan total penjualan cabang terendah sebesar 84.645.000 dengan jumlah transaksi sebanyak 1305. Dari temuan yang didapat 
pada bagian C, menu yang sebaiknya dipromosikan lebih gencar di cabang yogyakarta adalah menu-12 karena menu tersebut sudah terbukti
menjadi produk terlaris peringkat 1 di yogyakarta dengan kontribusi penjualan mencapai 9.275.000. Nilai ini bahkan lebih tinggi 
dibandingkan dengan menu peringkat 1 di Semarang dan Magelang. Meskipun secara omzet keseluruhan cabang yogyakarta paling rendah, 
pasar di sana sangat menyukai menu-12. Mendorong promosi atau bundling berbasis menu-12 dapat mendongkrak jumlah transaksi dan total
penjualan di cabang yogyakarta secara efektif

In [19]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
